# 6. METABRIC

Read this notebook as a checklist of what must *not* happen: no gene selection, no new
modules, no threshold chosen after seeing the result. The gene lists come from TCGA and are
only ever shrunk by what METABRIC failed to measure.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import Config
from src.io_utils import data_path, read_matrix, read_table, results_path

cfg = Config.load("../config.json")

In [ ]:
from src.modules import load_modules
from src.survival import design_matrix, prepare_endpoint
from src.validation import (
    concordance, module_coverage, module_preservation, restrict_modules,
    transfer_activity, validate_survival,
)

expr = read_matrix(data_path("metabric", "expression.tsv"))
clinical = read_table(data_path("metabric", "clinical.tsv"), index_col=0)
print(expr.shape)
clinical["CLAUDIN_SUBTYPE"].value_counts()

Microarray intensities, not counts, on a different scale entirely. That is why module
activity is a within-sample rank rather than a z-score: the rank is unchanged by any
monotone transform, so the same module means the same thing in both cohorts.

In [ ]:
tcga = read_matrix(results_path("preprocessed", "tcga_expression_raw.tsv.gz"))
pd.DataFrame({"TCGA": tcga.stack().describe(), "METABRIC": expr.stack().describe()}).round(2)

In [ ]:
modules = load_modules(results_path("modules", "modules.tsv"))
shortlist = [ln.strip() for ln in open(results_path("survival", "selected_modules.txt"))
             if ln.strip()]
modules = {m: g for m, g in modules.items() if m in shortlist}
module_coverage(modules, expr)

In [ ]:
modules = restrict_modules(modules, expr, min_coverage=0.5, min_size=5)
print(len(modules), "modules keep at least half their genes")

## Is the module still a module here

Before asking about survival, check the genes still co-express in METABRIC at all. A module
that falls apart on the other platform has nothing left to validate.

In [ ]:
module_preservation(expr, modules, n_perm=500, seed=cfg.seed)

In [ ]:
meta = pd.DataFrame({
    "age": pd.to_numeric(clinical["AGE_AT_DIAGNOSIS"], errors="coerce"),
    "grade": clinical["GRADE"].astype("string"),
    "subtype": clinical["CLAUDIN_SUBTYPE"].astype("string"),
    "OS": pd.to_numeric(clinical["OS"], errors="coerce"),
    "OS.time": pd.to_numeric(clinical["OS.time"], errors="coerce"),
})
surv = prepare_endpoint(meta, "OS", cfg.max_followup_days)
covariates = design_matrix(meta.loc[surv.index], ["age", "grade", "subtype"]).dropna()
print(len(surv), "patients,", int(surv["event"].sum()), "events")

In [ ]:
activity = transfer_activity(expr, modules)
result = validate_survival(activity, surv, covariates)
result[["module", "n", "events", "hr_uni", "ci_lower_uni", "ci_upper_uni", "p_uni", "q_uni"]]

In [ ]:
discovery = read_table(results_path("survival", "tcga_module_cox.tsv"))
concordance(discovery, result, alpha=cfg.fdr)

Replicated means the hazard ratio points the same way *and* stays significant. A module
that reverses direction has not been validated, however small its p-value is.